In [0]:
%sql
select * from gizmobox.bronze.v_customers
where customer_id is not null
order by customer_id;

In [0]:
%sql
select * from (select *, row_number() over(partition by customer_id order by customer_id) as rn
from gizmobox.bronze.v_customers
where customer_id is not null
order by customer_id)
 where rn >1;

In [0]:
%sql
select customer_id, customer_name from gizmobox.bronze.v_customers
where customer_id is not null
group by customer_id, customer_name
having count(customer_id) > 1
order by customer_id;

In [0]:
create table gizmobox.silver.customers
  as
with cte_distinct as(
select distinct * from gizmobox.bronze.v_customers
where customer_id is not null
order by customer_id),
cte_date as(
     select customer_id, max(created_timestamp) as max_Date
    from gizmobox.bronze.v_customers
    where customer_id is not null
    group by customer_id
)
select cast(d.created_timestamp as timestamp) as created_timestamp,
d.customer_id,
d.customer_name,
cast(d.date_of_birth as date) as date_of_birth,
d.email,
cast(d.member_since as date) as member_since,
d.telephone,
d.file_path,
d.file_name
from cte_distinct d
inner join cte_date t
on d.customer_id = t.customer_id
and d.created_timestamp = t.max_Date


